## Notebook to test changes to PyHealth Code for TPC-LoS

In [ ]:
from pyhealth.datasets import MIMIC4Dataset
import json
from pathlib import Path

### Load the data with modified MIMIC4Dataset

In [ ]:
DATASETS_ROOT = Path("~/Desktop/CS 598 - Deep Learning for Healthcare/datasets")

# MIMIC-IV Demo Dataset
MIMIC4_ROOT = DATASETS_ROOT / "mimic-iv-demo" / "2.2"

MIMIC4_TABLES = [
    "diagnoses_icd",
    "prescriptions",
    "labevents",
    "chartevents",
]

mimic4_base = MIMIC4Dataset(
    ehr_root=str(MIMIC4_ROOT),
    ehr_tables=MIMIC4_TABLES,
    dev=True,   # use 1000 patients while exploring
)

In [ ]:
#checking stats
print("Number of MIMIC patients:", len(mimic4_base.unique_patient_ids))
# inspecting one MIMIC patient and its loaded event partitions

first_patient = next(mimic4_base.iter_patients())

print("Patient type:", type(first_patient)) #info about patients
print("Patient fields:", list(vars(first_patient).keys()))
print("Patient dict:", vars(first_patient))


#see event partitions pyhealth created, which tables actually landed inside patient object, how each partition is stored
print("\nType of event_type_partitions:", type(first_patient.event_type_partitions))
print("Partition keys:", list(first_patient.event_type_partitions.keys()))

for key, value in first_patient.event_type_partitions.items():
    print(f"\nPartition: {key}")
    print("Type:", type(value))
    if isinstance(value, list):
        print("Length:", len(value))
        if len(value) > 0:
            print("First item type:", type(value[0]))
            print("First item:", value[0])
    elif isinstance(value, dict):
        print("Keys sample:", list(value.keys())[:10])
    else:
        print(value)

print("PARTITION SUMMARY")
for key, value in first_patient.event_type_partitions.items():
    if isinstance(value, list):
        print(f"{key}: list with {len(value)} events")
    elif isinstance(value, dict):
        print(f"{key}: dict with {len(value)} keys")
    else:
        print(f"{key}: {type(value)}")

#inspecting more patients
print("CHECKING MULTIPLE PATIENTS")
for i, patient in enumerate(mimic4_base.iter_patients()):
    print(f"\nPatient {i}")
    print("Patient fields:", list(vars(patient).keys()))
    print("Partition keys:", list(patient.event_type_partitions.keys()))
    if i == 2:
        break

#3# Testing chartevents works for specific patient

In [ ]:
patient = mimic4_base.get_patient(mimic4_base.unique_patient_ids[0])
events = patient.get_events(event_type="chartevents")
assert len(events) > 0
print(events[0].itemid, events[0].valuenum, events[0].timestamp)